In [24]:
import pandas as pd
import glob
import os

## 1) Create a single dataframe with the concatenation of all input csv files, adding a column called country

In [25]:
path = "./trendingYT/"
files = glob.glob(path + "*.csv.zst")

dfs = []

for f in files:
    country = os.path.basename(f)[:2]
    print("Loading:", f, "| Country:", country)

    df = pd.read_csv(
        f,
        compression="zstd",
        encoding="latin-1",
        on_bad_lines="skip", 
        engine="python"     
    )

    df["country"] = country
    dfs.append(df)

final_df = pd.concat(dfs, ignore_index=True)

print("Final shape:", final_df.shape)


Loading: ./trendingYT\CAvideos.csv.zst | Country: CA
Loading: ./trendingYT\DEvideos.csv.zst | Country: DE
Loading: ./trendingYT\FRvideos.csv.zst | Country: FR
Loading: ./trendingYT\GBvideos.csv.zst | Country: GB
Loading: ./trendingYT\INvideos.csv.zst | Country: IN
Loading: ./trendingYT\JPvideos.csv.zst | Country: JP
Loading: ./trendingYT\KRvideos.csv.zst | Country: KR
Loading: ./trendingYT\MXvideos.csv.zst | Country: MX
Loading: ./trendingYT\RUvideos.csv.zst | Country: RU
Loading: ./trendingYT\USvideos.csv.zst | Country: US
Final shape: (239800, 17)


In [26]:
final_df.head()

,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description,country
0,2kyS6SvSYSE,17.14.11,WE WANT TO TALK ABOUT OUR MARRIAGE,CaseyNeistat,22,2017-11-13T17:13:01.000Z,SHANtell martin,748374,57534,2967,15959,https://i.ytimg.com/vi/2kyS6SvSYSE/default.jpg,False,False,False,SHANTELL'S CHANNEL - https://www.youtube.com/s...,CA
1,JwboxqDylgg,17.14.11,Canada Soccer's Women's National Team v USA In...,Canada Soccer,17,2017-11-13T05:53:49.000Z,[none],36311,277,28,13,https://i.ytimg.com/vi/JwboxqDylgg/default.jpg,False,False,False,Canada Soccer's Women's National Team face riv...,CA
2,9B-q8h31Bpk,17.14.11,John Oliver Tackles Louis C.K. And Donald Trum...,TV Shows,22,2017-11-13T04:49:26.000Z,[none],106029,1270,101,181,https://i.ytimg.com/vi/9B-q8h31Bpk/default.jpg,False,False,False,"John Oliver on News, Politics ...",CA
3,1UE5Dq1rvUA,17.14.11,Taylor Swift Perform Ready For It - SNL,Ken Reactz,24,2017-11-12T05:18:02.000Z,[none],320964,8069,285,717,https://i.ytimg.com/vi/1UE5Dq1rvUA/default.jpg,False,False,False,Thanks for watching please subscribe and subsc...,CA
4,pmJQ4KwliX4,17.14.11,"LATEST Q POSTS: ROTHSCHILDS, HOUSE OF SAUD, lL...",James Munder,2,2017-11-12T21:25:40.000Z,[none],116820,1503,139,1066,https://i.ytimg.com/vi/pmJQ4KwliX4/default.jpg,False,False,False,https://pastebin.ca/3930472\n\nSupport My Chan...,CA


In [27]:
final_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 239800 entries, 0 to 239799
Data columns (total 17 columns):
 #   Column                  Non-Null Count   Dtype 
---  ------                  --------------   ----- 
 0   video_id                239800 non-null  object
 1   trending_date           221323 non-null  object
 2   title                   220697 non-null  object
 3   channel_title           220391 non-null  object
 4   category_id             219781 non-null  object
 5   publish_time            219485 non-null  object
 6   tags                    218502 non-null  object
 7   views                   218383 non-null  object
 8   likes                   218349 non-null  object
 9   dislikes                218305 non-null  object
 10  comment_count           218298 non-null  object
 11  thumbnail_link          218222 non-null  object
 12  comments_disabled       218211 non-null  object
 13  ratings_disabled        218059 non-null  object
 14  video_error_or_removed  218047 non-n

## 2) Extract all videos that have no tag

In [28]:
no_tag_df = final_df[
    final_df["tags"].isna() |
    (final_df["tags"].astype(str).str.strip() == "") |
    (final_df["tags"] == "[none]") |
    (final_df["tags"].str.lower() == "none")
]


## 3) For each channel, determine the total number of views

In [29]:
final_df['views'] = pd.to_numeric(final_df['views'], errors='coerce')
channel_views = final_df.groupby('channel_title')['views'].sum().reset_index()
channel_views = channel_views.sort_values(by='views', ascending=False)
channel_views


,channel_title,views
11812,Marvel Entertainment,3.003768e+09
17444,T-Series,2.299566e+09
21366,ibighit,1.799238e+09
17045,SpaceX,1.600641e+09
14466,PewDiePie,1.562112e+09
...,...,...
97,le prince William. PrÃ¨s de 15 minutes plus tard,0.000000e+00
96,la veuve du Taulier en voit dÃ©cidÃ©ment de t...,0.000000e+00
95,la premiÃ¨re nÃ©gociation entre les aÃ®nÃ©s d...,0.000000e+00
94,jade rollers,0.000000e+00


## 4) Save all rows with disabled comments and disabled ratings, or that have video_error_or_removed in a new dataframe called excluded, and remove those rows from the original dataframe.

In [30]:
# Filter rows to exclude
excluded = final_df[
    (final_df['comments_disabled'] == True) &
    (final_df['ratings_disabled'] == True) |
    (final_df['video_error_or_removed'] == True)
].copy()

final_df = final_df.drop(excluded.index)
# [~]

final_df = final_df.reset_index(drop=True)
excluded = excluded.reset_index(drop=True)

print("Excluded rows:", excluded.shape[0])
print("Remaining rows in final_df:", final_df.shape[0])


Excluded rows: 1473
Remaining rows in final_df: 238327


## 5) Add a like_ratio column storing the ratio between the number of likes and of dislikes

In [31]:
final_df['likes'] = pd.to_numeric(final_df['likes'], errors='coerce')
final_df['dislikes'] = pd.to_numeric(final_df['dislikes'], errors='coerce')
final_df['like_ratio'] = final_df['likes'] / final_df['dislikes'].replace(0, pd.NA)
final_df['like_ratio'] = final_df['like_ratio'].fillna(final_df['likes'])
final_df[['likes', 'dislikes', 'like_ratio']].head()


C:\Users\User\AppData\Local\Temp\ipykernel_15112\787068149.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  final_df['like_ratio'] = final_df['like_ratio'].fillna(final_df['likes'])


,likes,dislikes,like_ratio
0,57534.0,2967.0,19.391304
1,277.0,28.0,9.892857
2,1270.0,101.0,12.574257
3,8069.0,285.0,28.312281
4,1503.0,139.0,10.812950


## 6) Cluster the publish time into 10-minute intervals (e.g. from 02:20 to 02:30)

In [32]:
final_df['publish_time'] = pd.to_datetime(final_df['publish_time'], errors='coerce')
final_df['publish_time_10min'] = final_df['publish_time'].dt.floor('10min')
final_df['publish_time_10min']

0        2017-11-13 17:10:00+00:00
1        2017-11-13 05:50:00+00:00
2        2017-11-13 04:40:00+00:00
3        2017-11-12 05:10:00+00:00
4        2017-11-12 21:20:00+00:00
                    ...           
238322   2018-05-29 12:00:00+00:00
238323                         NaT
238324                         NaT
238325                         NaT
238326   2018-05-18 01:00:00+00:00
Name: publish_time_10min, Length: 238327, dtype: datetime64[ns, UTC]

## 7) For each interval, determine the number of videos, average number of likes and of dislikes.

In [45]:
interval_stats = (
    final_df
    .groupby('publish_time_10min')
    .agg(
        videos_count = ('video_id', 'count'),
        avg_likes    = ('likes', 'mean'),
        avg_dislikes = ('dislikes', 'mean')
    )
    .reset_index()
)


In [48]:
interval_stats

,publish_time_10min,videos_count,avg_likes,avg_dislikes
0,2009-03-22 19:50:00+00:00,1,513.0,10.0
1,2009-07-23 17:20:00+00:00,1,42.0,1.0
2,2009-10-07 09:20:00+00:00,1,3109.0,13.0
3,2009-10-31 22:10:00+00:00,1,2.0,1.0
4,2010-01-03 23:40:00+00:00,2,883.5,19.0
...,...,...,...,...
29307,2018-06-14 02:30:00+00:00,1,853.0,77.0
29308,2018-06-14 03:00:00+00:00,2,2304.5,31.5
29309,2018-06-14 03:20:00+00:00,1,1414.0,28.0
29310,2018-06-14 03:30:00+00:00,1,8481.0,252.0


## 8) For each tag, determine the number of videos

In [34]:
final_df['tags'] = final_df['tags'].astype(str)

final_df['tags'] = final_df['tags'].replace({'[none]': None})

final_df['tag_list'] = final_df['tags'].str.split('|')

exploded_tags = final_df.explode('tag_list')

exploded_tags = exploded_tags[
    exploded_tags['tag_list'].notna() & (exploded_tags['tag_list'].str.strip() != "")
]

tag_counts = (
    exploded_tags.groupby('tag_list')['video_id']
    .nunique()
    .reset_index(name='video_count')
    .sort_values('video_count', ascending=False)
)

tag_counts.head()


,tag_list,video_count
9313,"""2018""",3890
214648,"""funny""",1827
8858,"""2017""",1805
597340,None,1768
325485,"""show""",1442


## 9) Find the tags with the largest number of videos

In [35]:
final_df['tags'] = final_df['tags'].astype(str)

final_df['tags'] = final_df['tags'].replace({'[none]': None})

final_df['tag_list'] = final_df['tags'].str.split('|')

exploded_tags = final_df.explode('tag_list')

exploded_tags = exploded_tags[
    exploded_tags['tag_list'].notna() & (exploded_tags['tag_list'].str.strip() != "")
]

tag_counts = (
    exploded_tags.groupby('tag_list')['video_id']
    .nunique()
    .reset_index(name='video_count')
)

top_tags = tag_counts.sort_values(by='video_count', ascending=False)

top_tags.head()


,tag_list,video_count
597340,None,25542
9313,"""2018""",3890
214648,"""funny""",1827
8858,"""2017""",1805
325485,"""show""",1442


## 10) For each (tag, country) pair, compute average ratio likes/dislikes

In [36]:
final_df['likes'] = pd.to_numeric(final_df['likes'], errors='coerce')
final_df['dislikes'] = pd.to_numeric(final_df['dislikes'], errors='coerce')

final_df['like_ratio'] = final_df['likes'] / final_df['dislikes'].replace(0, pd.NA)
final_df['like_ratio'] = final_df['like_ratio'].fillna(final_df['likes'])  

final_df['tags'] = final_df['tags'].astype(str)
final_df['tags'] = final_df['tags'].replace({'[none]': None})

final_df['tag_list'] = final_df['tags'].str.split('|')

tag_country = final_df.explode('tag_list')

tag_country = tag_country[
    tag_country['tag_list'].notna() & (tag_country['tag_list'].str.strip() != "")
]

tag_country_ratio = (
    tag_country.groupby(['tag_list', 'country'])['like_ratio']
    .mean()
    .reset_index(name='avg_like_ratio')
)

tag_country_ratio.head(20)


C:\Users\User\AppData\Local\Temp\ipykernel_15112\2770094056.py:5: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  final_df['like_ratio'] = final_df['like_ratio'].fillna(final_df['likes'])


,tag_list,country,avg_like_ratio
0,"#Freeticket""",IN,2.665679
1,#GST,IN,2.416896
2,#Jaisimha,IN,2.665679
3,#JanaSenaParty,IN,17.621907
4,"#MahaaNews""",IN,8.511947
5,#PawanKalyan #AlluArjun,IN,9.983453
6,"#PrimeTimeWithMurthy""",IN,6.263998
7,#RamGopalVarma,IN,2.416896
8,#VMA 20,FR,9.615385
9,#VMA 21,FR,5.782258


## 11) For each (trending_date, country) pair, the video with the largest number of views

In [37]:
df_sorted = df.sort_values(['trending_date', 'country', 'views'], 
                           ascending=[True, True, False])

result = df_sorted.groupby(['trending_date', 'country']).head(1)

result = result.reset_index(drop=True)

print(result)

                                              video_id  \
0                 \nSet for release beginning March 16   
1    Black Panther is available on Blu-Ray and DVD ...   
2                   \nTomb Raider in theaters March 16   
3                    \nJames Ambler and Keith Glickman   
4                                              \nToday   
..                                                 ...   
274                                        t3z_pdr6z8w   
275                                        p-bGJ6W1Im4   
276                                        1h7KV2sjUWY   
277  \nâTomb Raiderâ also stars Dominic West (â...   
278  \nUthaug directed from a script by Geneva Robe...   

                                         trending_date  \
0                                                 2018   
1                                                2018.   
2     2018\n\nStill havenât subscribed to WIRED o...   
3                                            37 and 29   
4     AP empl

## 12) Divide trending_date into three columns: year, month, day

In [38]:
df['trending_date'] = pd.to_datetime(df['trending_date'], errors='coerce')

df['year'] = df['trending_date'].dt.year
df['month'] = df['trending_date'].dt.month
df['day'] = df['trending_date'].dt.day

print(df[['trending_date', 'year', 'month', 'day']].head())


  trending_date  year  month  day
0           NaT   NaN    NaN  NaN
1           NaT   NaN    NaN  NaN
2           NaT   NaN    NaN  NaN
3           NaT   NaN    NaN  NaN
4           NaT   NaN    NaN  NaN


C:\Users\User\AppData\Local\Temp\ipykernel_15112\3036582384.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['trending_date'] = pd.to_datetime(df['trending_date'], errors='coerce')
